In [1]:
import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import ccxt
from itertools import combinations
from statsmodels.tsa.stattools import coint
from scipy.stats import ttest_rel

pd.set_option("display.width", 140)

In [2]:
UNIVERSE = [
    "BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA", "LINK", "AVAX", "DOT",
    "LTC", "ATOM", "AAVE", "UNI", "ARB", "OP", "SUI", "NEAR", "ALGO", "FIL"
]
SYMBOLS = [f"{coin}/USD" for coin in UNIVERSE]

In [3]:
exchange = ccxt.coinbase()
markets = exchange.load_markets()

symbols = [f"{coin}/USD" for coin in UNIVERSE if f"{coin}/USD" in markets]
pairs = list(combinations(symbols, 2))
print(f"{len(symbols)} coins available, {len(pairs)} candidate pairs")

20 coins available, 190 candidate pairs


In [4]:
def to_df(data, symbol):
    df = pd.DataFrame(data, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df[["timestamp", "close", "volume"]].copy()
    df = df.rename(columns={"close": symbol, "volume": f"{symbol}_volume"})
    return df

In [5]:
exchange = ccxt.coinbase({
    "enableRateLimit": True,
    "timeout": 30000,  
})

def fetch_ohlcv_history(exchange, symbol, timeframe, since_ms, limit=300, max_retries=5):
    all_rows = []
    since = since_ms
    now_ms = exchange.milliseconds()

    while since < now_ms:
        batch = None
        for attempt in range(max_retries):
            try:
                batch = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=limit)
                break
            except (ccxt.RequestTimeout, ccxt.NetworkError) as e:
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)

        if not batch:
            break

        all_rows.extend(batch)
        since = batch[-1][0] + 1

    return all_rows

In [23]:
YEARS_OF_HISTORY = 5
TIMEFRAME = "1h"

since_dt = pd.Timestamp.now(tz="UTC") - pd.DateOffset(years=YEARS_OF_HISTORY)
HISTORY_SINCE = exchange.parse8601(since_dt.strftime("%Y-%m-%dT%H:%M:%SZ"))

hourly_prices = None
for coin in UNIVERSE:
    symbol = f"{coin}/USD"
    if symbol not in markets:
        continue
    data = fetch_ohlcv_history(exchange, symbol, timeframe=TIMEFRAME, since_ms=HISTORY_SINCE)
    if not data:
        continue
    df = to_df(data, coin)
    hourly_prices = df if hourly_prices is None else hourly_prices.merge(df, on="timestamp", how="inner")

hourly_prices = hourly_prices.sort_values("timestamp").reset_index(drop=True)
print(f"{len(hourly_prices)} aligned hourly bars, {hourly_prices['timestamp'].min()} -> {hourly_prices['timestamp'].max()}")

/var/folders/74/lqx4_3w16sz74ftgv_mn1ytr0000gn/T/ipykernel_17468/1872878780.py:4: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  since_dt = pd.Timestamp.utcnow() - pd.DateOffset(years=YEARS_OF_HISTORY)


43808 aligned hourly bars, 2021-08-04 21:00:00 -> 2026-08-04 20:00:00


In [ ]:
# Research configuration and immutable train/test split

# The split timestamp is defined ONCE from the master hourly sample.
# Every 4h/8h/12h experiment uses the same calendar train/test boundary.
TRAIN_FRAC = 0.70
_SPLIT_IDX_1H = int(len(hourly_prices) * TRAIN_FRAC)
TRAIN_END = hourly_prices.loc[_SPLIT_IDX_1H, "timestamp"]
TEST_START = TRAIN_END + pd.Timedelta(hours=1)

CV_FOLDS = 4
CV_TRAIN_FRACTION = 0.60
CV_MIN_TRAIN_BARS = 400
CV_MIN_VALIDATE_BARS = 100

SAMPLING_HOURS = [4, 8, 12]
HALF_LIFE_MULTIPLIERS = [0.50, 0.75, 1.00, 1.25, 1.50, 2.00]
MAX_HALF_LIFE_DAYS = 30

FEE_BPS = 20.0
MIN_CORR = 0.80
MAX_COINTEGRATION_P = 0.10
MIN_NET_SHARPE = 0.0
MAX_HEDGE_RATIO_CV = 0.25
MIN_CONVERGENCES_PER_YEAR = 4.0
MAX_PAIRS = 8

LOW_VOLUME_RATIO = 0.75
HIGH_VOLUME_RATIO = 1.50
LOW_VOLUME_BOOST = 1.20
HIGH_VOLUME_SCALE = 0.00
VOLUME_WINDOW_HOURS = 7 * 24

# Keep signal thresholds fixed for this first CV so the requested sampling/window
# comparison is not buried inside a large hyperparameter search.
STRATEGY_PARAMS = {
    "zscore": {"entry_z": 2.5, "exit_z": 0.5},
    "bollinger": {"band_k": 1.5},
}

print(f"Master split: train through {TRAIN_END}, test begins {TEST_START}")

In [ ]:
# Sampling and alignment helpers

def sample_market_data(hourly_df, sampling_hours):
    """Resample hourly OHLCV correctly: close=last, volume=sum."""
    df = hourly_df.copy().sort_values("timestamp").set_index("timestamp")
    price_cols = [c for c in UNIVERSE if c in df.columns]
    volume_cols = [f"{c}_volume" for c in UNIVERSE if f"{c}_volume" in df.columns]
    agg = {c: "last" for c in price_cols}
    agg.update({c: "sum" for c in volume_cols})
    out = df.resample(f"{sampling_hours}h").agg(agg).dropna(how="all").reset_index()
    return out

def split_at_fixed_boundary(sampled_df):
    train = sampled_df[sampled_df["timestamp"] <= TRAIN_END].copy().reset_index(drop=True)
    test = sampled_df[sampled_df["timestamp"] > TRAIN_END].copy().reset_index(drop=True)
    return train, test

def align_pair_prices(prices_df, asset_a, asset_b):
    cols = ["timestamp", asset_a, asset_b, f"{asset_a}_volume", f"{asset_b}_volume"]
    cols = [c for c in cols if c in prices_df.columns]
    return prices_df[cols].dropna().sort_values("timestamp").reset_index(drop=True)

In [ ]:
# Pair diagnostics

def estimate_ols_spread(df, asset_a, asset_b):
    out = df.copy()
    out["log_a"] = np.log(out[asset_a])
    out["log_b"] = np.log(out[asset_b])
    model = sm.OLS(out["log_a"], sm.add_constant(out["log_b"])).fit()
    alpha = float(model.params["const"])
    beta = float(model.params["log_b"])
    out["spread"] = model.resid
    corr = float(out[["log_a", "log_b"]].corr().iloc[0, 1])
    _, eg_pvalue, _ = coint(out["log_a"], out["log_b"])
    return out, alpha, beta, corr, float(eg_pvalue)

def apply_fixed_spread(df, asset_a, asset_b, alpha, beta):
    out = df.copy()
    out["log_a"] = np.log(out[asset_a])
    out["log_b"] = np.log(out[asset_b])
    out["spread"] = out["log_a"] - (alpha + beta * out["log_b"])
    return out

def spread_half_life_bars(spread):
    s = pd.Series(spread).dropna()
    tmp = pd.concat([s.diff(), s.shift(1)], axis=1).dropna()
    if len(tmp) < 20:
        return np.nan
    tmp.columns = ["diff", "lag"]
    lam = sm.OLS(tmp["diff"], sm.add_constant(tmp["lag"])).fit().params["lag"]
    if lam >= 0:
        return np.nan
    return float(-np.log(2) / lam)

def rolling_hedge_ratio_cv(df, asset_a, asset_b, window_bars):
    if len(df) < window_bars * 2:
        return np.nan
    a = np.log(df[asset_a])
    b = np.log(df[asset_b])
    betas = []
    step = max(1, window_bars // 4)
    for end in range(window_bars, len(df) + 1, step):
        y = a.iloc[end-window_bars:end]
        x = sm.add_constant(b.iloc[end-window_bars:end])
        betas.append(sm.OLS(y, x).fit().params.iloc[1])
    betas = pd.Series(betas).dropna()
    if len(betas) < 3 or abs(betas.mean()) < 1e-12:
        return np.nan
    return float(betas.std() / abs(betas.mean()))

def convergence_count_per_year(z, sampling_hours, threshold=2.0):
    z = pd.Series(z).dropna()
    if len(z) < 2:
        return 0.0
    hits = int(((z.shift(1) < threshold) & (z >= threshold)).sum() + ((z.shift(1) > -threshold) & (z <= -threshold)).sum())
    years = max((len(z) * sampling_hours) / (24 * 365), 1 / 365)
    return float(hits / years)

In [ ]:
# Reversal signal functions

def rolling_zscore(series, window_bars):
    m = series.rolling(window_bars, min_periods=window_bars).mean()
    s = series.rolling(window_bars, min_periods=window_bars).std()
    return (series - m) / s.replace(0, np.nan)

def build_positions_from_signal(z, entry_z, exit_z, min_persist=1, cooldown_bars=0, allow_entry=None):
    position = 0
    cooldown = 0
    below = 0
    above = 0
    out = []
    for i, value in enumerate(z):
        if pd.isna(value):
            out.append(0); continue
        below = below + 1 if value < -entry_z else 0
        above = above + 1 if value > entry_z else 0
        cooldown = max(cooldown - 1, 0)
        ok = True if allow_entry is None else bool(allow_entry.iloc[i])
        if position == 0 and cooldown == 0 and ok:
            if below >= min_persist:
                position = 1
            elif above >= min_persist:
                position = -1
        elif position == 1 and value > -exit_z:
            position = 0; cooldown = cooldown_bars
        elif position == -1 and value < exit_z:
            position = 0; cooldown = cooldown_bars
        out.append(position)
    return pd.Series(out, index=z.index)

def build_bollinger_positions(spread, window_bars, band_k, min_persist=1, cooldown_bars=0, allow_entry=None):
    mid = spread.rolling(window_bars, min_periods=window_bars).mean()
    std = spread.rolling(window_bars, min_periods=window_bars).std()
    upper = mid + band_k * std
    lower = mid - band_k * std
    # Reuse the same state machine as z-score, with exits at the rolling mean.
    position = 0; cooldown = 0; below = 0; above = 0; out = []
    for i, (s, m, u, l) in enumerate(zip(spread, mid, upper, lower)):
        if any(pd.isna(v) for v in (s, m, u, l)):
            out.append(0); continue
        below = below + 1 if s < l else 0
        above = above + 1 if s > u else 0
        cooldown = max(cooldown - 1, 0)
        ok = True if allow_entry is None else bool(allow_entry.iloc[i])
        if position == 0 and cooldown == 0 and ok:
            if below >= min_persist: position = 1
            elif above >= min_persist: position = -1
        elif position == 1 and s >= m:
            position = 0; cooldown = cooldown_bars
        elif position == -1 and s <= m:
            position = 0; cooldown = cooldown_bars
        out.append(position)
    return pd.Series(out, index=spread.index)

In [ ]:
# Leakage-safe backtest

def position_volume_scale(pair_df, volume_window_bars, low_ratio=LOW_VOLUME_RATIO, high_ratio=HIGH_VOLUME_RATIO, low_boost=LOW_VOLUME_BOOST, high_scale=HIGH_VOLUME_SCALE):
    va, vb = f"{pair_df.attrs['asset_a']}_volume", f"{pair_df.attrs['asset_b']}_volume"
    if va not in pair_df.columns or vb not in pair_df.columns:
        return pd.Series(1.0, index=pair_df.index)
    pair_volume = pair_df[va] + pair_df[vb]
    med = pair_volume.rolling(volume_window_bars, min_periods=volume_window_bars).median()
    ratio = pair_volume / med.replace(0, np.nan)
    scale = pd.Series(1.0, index=pair_df.index)
    scale = scale.where(~(ratio <= low_ratio), low_boost)
    scale = scale.where(~(ratio >= high_ratio), high_scale)
    return scale.fillna(1.0)

def backtest_pair(prices_df, asset_a, asset_b, alpha, beta, strategy, window_bars, sampling_hours, fee_bps=FEE_BPS, entry_z=2.0, exit_z=0.25, band_k=1.5, min_persist=1, cooldown_bars=0, volume_window_bars=42):
    df = align_pair_prices(prices_df, asset_a, asset_b)
    df = apply_fixed_spread(df, asset_a, asset_b, alpha, beta)
    df.attrs['asset_a'] = asset_a; df.attrs['asset_b'] = asset_b
    volume_scale = position_volume_scale(df, volume_window_bars)
    volume_ratio = volume_scale.copy()
    # The scale itself encodes the mentor's low-volume preference:
    # low volume -> slightly larger position; high volume -> no entry/zero exposure.
    if strategy == "zscore":
        signal = rolling_zscore(df["spread"], window_bars)
        position = build_positions_from_signal(signal, entry_z=entry_z, exit_z=exit_z, min_persist=min_persist, cooldown_bars=cooldown_bars)
    elif strategy == "bollinger":
        position = build_bollinger_positions(df["spread"], window_bars, band_k=band_k, min_persist=min_persist, cooldown_bars=cooldown_bars)
    else:
        raise ValueError("strategy must be 'zscore' or 'bollinger'")
    df["position"] = position * volume_ratio
    gross = 1.0 + abs(beta)
    df["w_a"] = df["position"] / gross
    df["w_b"] = -beta * df["position"] / gross
    df["ret_a"] = df[asset_a].pct_change()
    df["ret_b"] = df[asset_b].pct_change()
    df["turnover"] = df["w_a"].diff().abs().fillna(0) + df["w_b"].diff().abs().fillna(0)
    raw = df["w_a"].shift(1) * df["ret_a"] + df["w_b"].shift(1) * df["ret_b"]
    df["strategy_ret"] = raw - df["turnover"] * (fee_bps / 10000.0)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    ann = np.sqrt((24 * 365) / sampling_hours)
    r = df["strategy_ret"].dropna()
    sharpe = np.nan if r.std() == 0 or pd.isna(r.std()) else r.mean() / r.std() * ann
    equity = (1 + r.fillna(0)).cumprod()
    max_dd = (equity / equity.cummax() - 1).min() if len(equity) else np.nan
    metrics = {
        "Sharpe": sharpe,
        "NetReturn": equity.iloc[-1] - 1 if len(equity) else np.nan,
        "MaxDD": max_dd,
        "Turnover": df["turnover"].mean(),
        "Trades": int((df["position"].diff().fillna(0).abs() > 1e-12).sum()),
    }
    return df, pd.Series(metrics)

def portfolio_metrics(ret, sampling_hours):
    r = ret.dropna()
    ann = np.sqrt((24 * 365) / sampling_hours)
    vol = r.std()
    equity = (1 + r.fillna(0)).cumprod()
    return {
        "Sharpe": np.nan if vol == 0 or pd.isna(vol) else r.mean() / vol * ann,
        "NetReturn": equity.iloc[-1] - 1 if len(equity) else np.nan,
        "MaxDD": (equity / equity.cummax() - 1).min() if len(equity) else np.nan,
        "Turnover": np.nan,
    }

In [ ]:
# Training-only pair selection

def pair_diagnostics(train_df, asset_a, asset_b, sampling_hours, signal_window_bars, strategy, params):
    pair = align_pair_prices(train_df, asset_a, asset_b)
    if len(pair) < max(signal_window_bars * 3, 200):
        return None
    fitted, alpha, beta, corr, pval = estimate_ols_spread(pair, asset_a, asset_b)
    spread = fitted["spread"]
    hl_bars = spread_half_life_bars(spread)
    hl_hours = hl_bars * sampling_hours if pd.notna(hl_bars) else np.nan
    stability = rolling_hedge_ratio_cv(pair, asset_a, asset_b, max(signal_window_bars * 4, 100))
    z = rolling_zscore(spread, signal_window_bars)
    conv = convergence_count_per_year(z, sampling_hours, threshold=2.0)
    if strategy == "zscore":
        bt, m = backtest_pair(train_df, asset_a, asset_b, alpha, beta, strategy, signal_window_bars, sampling_hours, fee_bps=FEE_BPS, entry_z=params["entry_z"], exit_z=params["exit_z"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // sampling_hours))
    else:
        bt, m = backtest_pair(train_df, asset_a, asset_b, alpha, beta, strategy, signal_window_bars, sampling_hours, fee_bps=FEE_BPS, band_k=params["band_k"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // sampling_hours))
    return {
        "Pair": f"{asset_a}/{asset_b}", "Asset_A": asset_a, "Asset_B": asset_b,
        "Corr": corr, "EG_Coint_Pvalue": pval, "Alpha": alpha, "Beta": beta,
        "HalfLifeHours": hl_hours, "SpreadVol": spread.std(),
        "HedgeRatioCV": stability, "ConvergencesPerYear": conv,
        "TrainNetSharpe": m["Sharpe"], "TrainTurnover": m["Turnover"],
    }

def select_pairs(train_df, sampling_hours, signal_window_bars, strategy, params, max_pairs=MAX_PAIRS):
    rows = []
    for a, b in combinations([c for c in UNIVERSE if c in train_df.columns], 2):
        try:
            d = pair_diagnostics(train_df, a, b, sampling_hours, signal_window_bars, strategy, params)
            if d is not None:
                rows.append(d)
        except Exception:
            continue
    tab = pd.DataFrame(rows)
    if tab.empty:
        return tab
    tab = tab[(tab["Corr"] > MIN_CORR) & (tab["EG_Coint_Pvalue"] < MAX_COINTEGRATION_P)]
    tab = tab[tab["HalfLifeHours"].between(2 * sampling_hours, MAX_HALF_LIFE_DAYS * 24, inclusive="both")]
    tab = tab[tab["HedgeRatioCV"].notna() & (tab["HedgeRatioCV"] <= MAX_HEDGE_RATIO_CV)]
    tab = tab[tab["ConvergencesPerYear"] >= MIN_CONVERGENCES_PER_YEAR]
    tab = tab[tab["TrainNetSharpe"].fillna(-np.inf) > MIN_NET_SHARPE]
    if tab.empty:
        return tab
    # Rank on training-only quality; do not use the validation fold here.
    tab["SelectionScore"] = (
        tab["TrainNetSharpe"] / (1 + 20 * tab["TrainTurnover"].fillna(0))
        / (1 + tab["HedgeRatioCV"].fillna(MAX_HEDGE_RATIO_CV))
    )
    return tab.sort_values(["SelectionScore", "TrainNetSharpe"], ascending=False).head(max_pairs).reset_index(drop=True)

In [ ]:
# Walk-forward cross-validation

def inverse_vol_weights(ret_series):
    rets = pd.concat(ret_series, axis=1).dropna(how="all")
    vols = rets.std().replace(0, np.nan)
    w = 1 / vols
    return (w / w.sum()).fillna(0.0)

def enhanced_weights(ret_series, pair_metrics):
    rets = pd.concat(ret_series, axis=1).dropna(how="all")
    vol = rets.std().replace(0, np.nan)
    sharpe = pair_metrics["TrainNetSharpe"].reindex(rets.columns).clip(lower=0).fillna(0)
    turnover = pair_metrics["TrainTurnover"].reindex(rets.columns).fillna(0)
    base = sharpe / vol
    cost_penalty = 1 / (1 + 20 * turnover)
    score = base * cost_penalty
    # Penalize redundant pairs using training-period return correlation only.
    corr = rets.corr().abs().fillna(0)
    for p in score.index:
        others = [q for q in score.index if q != p and score.get(q, 0) > 0]
        if others:
            score.loc[p] /= (1 + corr.loc[p, others].mean())
    return (score / score.sum()).fillna(0.0) if score.sum() > 0 else inverse_vol_weights(ret_series)

def make_fold_slices(train_df, n_folds=CV_FOLDS):
    n = len(train_df)
    first_train = max(CV_MIN_TRAIN_BARS, int(n * CV_TRAIN_FRACTION))
    remaining = n - first_train
    step = remaining // n_folds
    folds = []
    for i in range(n_folds):
        train_end = first_train + i * step
        val_end = first_train + (i + 1) * step if i < n_folds - 1 else n
        if train_end < CV_MIN_TRAIN_BARS or val_end - train_end < CV_MIN_VALIDATE_BARS:
            continue
        folds.append((train_df.iloc[:train_end].copy(), train_df.iloc[train_end:val_end].copy()))
    return folds

def cv_one_config(train_df, sampling_hours, window_multiplier, strategy, params, weight_mode="enhanced"):
    sampled_train = sample_market_data(train_df, sampling_hours)
    folds = make_fold_slices(sampled_train)
    fold_rows = []
    for fold_id, (fit_df, val_df) in enumerate(folds, 1):
        # Estimate the window scale from the CURRENT fitting sample only.
        hls = []
        for a, b in combinations([c for c in UNIVERSE if c in fit_df.columns], 2):
            try:
                pair = align_pair_prices(fit_df, a, b)
                if len(pair) >= 200:
                    fitted, *_ = estimate_ols_spread(pair, a, b)
                    hl = spread_half_life_bars(fitted["spread"])
                    if pd.notna(hl): hls.append(hl * sampling_hours)
            except Exception:
                pass
        mean_hl_hours = float(np.nanmean(hls)) if hls else np.nan
        if not np.isfinite(mean_hl_hours):
            continue
        window_hours = max(float(sampling_hours), window_multiplier * mean_hl_hours)
        window_hours = round(window_hours / sampling_hours) * sampling_hours
        window_bars = max(2, int(round(window_hours / sampling_hours)))
        selected = select_pairs(fit_df, sampling_hours, window_bars, strategy, params)
        if selected.empty:
            fold_rows.append({"Fold": fold_id, "SamplingHours": sampling_hours, "WindowMultiplier": window_multiplier, "WindowHours": window_hours, "Strategy": strategy, "Sharpe": np.nan, "NetReturn": np.nan, "Pairs": 0, "MeanHalfLifeHours": mean_hl_hours})
            continue
        ret_series = {}; train_pair_metrics = []
        warmup = max(window_bars, VOLUME_WINDOW_HOURS // sampling_hours + 2)
        val_start = val_df["timestamp"].min()
        bt_input = pd.concat([fit_df.tail(warmup), val_df], ignore_index=True).drop_duplicates("timestamp")
        for _, row in selected.iterrows():
            a, b = row["Asset_A"], row["Asset_B"]
            if strategy == "zscore":
                bt, m = backtest_pair(fit_df, a, b, row["Alpha"], row["Beta"], strategy, window_bars, sampling_hours, fee_bps=FEE_BPS, entry_z=params["entry_z"], exit_z=params["exit_z"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // sampling_hours))
                bt_val, _ = backtest_pair(bt_input, a, b, row["Alpha"], row["Beta"], strategy, window_bars, sampling_hours, fee_bps=FEE_BPS, entry_z=params["entry_z"], exit_z=params["exit_z"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // sampling_hours))
            else:
                bt, m = backtest_pair(fit_df, a, b, row["Alpha"], row["Beta"], strategy, window_bars, sampling_hours, fee_bps=FEE_BPS, band_k=params["band_k"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // sampling_hours))
                bt_val, _ = backtest_pair(bt_input, a, b, row["Alpha"], row["Beta"], strategy, window_bars, sampling_hours, fee_bps=FEE_BPS, band_k=params["band_k"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // sampling_hours))
            pair_name = row["Pair"]
            ret_series[pair_name] = bt["strategy_ret"].rename(pair_name)
            train_pair_metrics.append(row)
        pair_metrics = pd.DataFrame(train_pair_metrics).set_index("Pair")
        weights = inverse_vol_weights(ret_series) if weight_mode == "equal_vol" else enhanced_weights(ret_series, pair_metrics)
        val_rets = {}
        for _, row in selected.iterrows():
            a, b = row["Asset_A"], row["Asset_B"]
            if strategy == "zscore":
                bt_val, _ = backtest_pair(bt_input, a, b, row["Alpha"], row["Beta"], strategy, window_bars, sampling_hours, fee_bps=FEE_BPS, entry_z=params["entry_z"], exit_z=params["exit_z"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // sampling_hours))
            else:
                bt_val, _ = backtest_pair(bt_input, a, b, row["Alpha"], row["Beta"], strategy, window_bars, sampling_hours, fee_bps=FEE_BPS, band_k=params["band_k"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // sampling_hours))
            val_rets[row["Pair"]] = bt_val.loc[bt_val.index >= val_start, "strategy_ret"]
        val_mat = pd.concat(val_rets, axis=1).dropna(how="all")
        w = weights.reindex(val_mat.columns).fillna(0)
        if w.sum() > 0: w = w / w.sum()
        port = val_mat.mul(w, axis=1).sum(axis=1).dropna()
        pm = portfolio_metrics(port, sampling_hours)
        fold_rows.append({"Fold": fold_id, "SamplingHours": sampling_hours, "WindowMultiplier": window_multiplier, "WindowHours": window_hours, "Strategy": strategy, "WeightMode": weight_mode, "Sharpe": pm["Sharpe"], "NetReturn": pm["NetReturn"], "MaxDD": pm["MaxDD"], "Pairs": len(selected), "MeanHalfLifeHours": mean_hl_hours})
    return pd.DataFrame(fold_rows)

In [ ]:
# Run ONLY on the original training period.
# CV dimensions: sampling frequency x half-life window multiplier x strategy x weighting.
# Every fold re-estimates half-life, hedge ratios, pair eligibility, and portfolio weights
# using fit_df only; the following validation block is never used in selection.
cv_rows = []
for sampling_hours in SAMPLING_HOURS:
    for window_multiplier in HALF_LIFE_MULTIPLIERS:
        for strategy, params in STRATEGY_PARAMS.items():
            for weight_mode in ["equal_vol", "enhanced"]:
                folds = cv_one_config(
                    train_df=hourly_prices[hourly_prices["timestamp"] <= TRAIN_END].copy(),
                    sampling_hours=sampling_hours,
                    window_multiplier=window_multiplier,
                    strategy=strategy,
                    params=params,
                    weight_mode=weight_mode,
                )
                if folds.empty:
                    continue
                row = folds.mean(numeric_only=True).to_dict()
                row.update({
                    "SamplingHours": sampling_hours,
                    "WindowMultiplier": window_multiplier,
                    "Strategy": strategy,
                    "WeightMode": weight_mode,
                    **params,
                    "ValidFolds": folds["Fold"].nunique(),
                })
                row["SharpeStd"] = folds["Sharpe"].std()
                row["PositiveFoldFraction"] = (folds["Sharpe"] > 0).mean()
                cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values(
    ["Sharpe", "PositiveFoldFraction"], ascending=False
).reset_index(drop=True)
print(f"CV configurations evaluated: {len(cv_results)}")
display(cv_results.head(20))

In [ ]:
# Freeze strategy before touching the test set

# Select exactly ONE configuration using CV results.
best_cv = cv_results.iloc[0].copy()
print("Selected from training CV only:")
print(best_cv)

FINAL_SAMPLING_HOURS = int(best_cv["SamplingHours"])
FINAL_WINDOW_MULTIPLIER = float(best_cv["WindowMultiplier"])
FINAL_STRATEGY = best_cv["Strategy"]
FINAL_WEIGHT_MODE = best_cv["WeightMode"]
FINAL_PARAMS = ({"entry_z": float(best_cv["entry_z"]), "exit_z": float(best_cv["exit_z"])} if FINAL_STRATEGY == "zscore" else {"band_k": float(best_cv["band_k"])})

# Re-estimate the final signal window and pair selection on ALL TRAIN data only.
final_sampled_train = sample_market_data(hourly_prices[hourly_prices["timestamp"] <= TRAIN_END], FINAL_SAMPLING_HOURS)
hls = []
for a, b in combinations([c for c in UNIVERSE if c in final_sampled_train.columns], 2):
    try:
        pair = align_pair_prices(final_sampled_train, a, b)
        fitted, *_ = estimate_ols_spread(pair, a, b)
        hl = spread_half_life_bars(fitted["spread"])
        if pd.notna(hl): hls.append(hl * FINAL_SAMPLING_HOURS)
    except Exception:
        pass
FINAL_MEAN_HALF_LIFE_HOURS = float(np.nanmean(hls))
FINAL_WINDOW_HOURS = round(max(FINAL_SAMPLING_HOURS, FINAL_WINDOW_MULTIPLIER * FINAL_MEAN_HALF_LIFE_HOURS) / FINAL_SAMPLING_HOURS) * FINAL_SAMPLING_HOURS
FINAL_WINDOW_BARS = max(2, int(FINAL_WINDOW_HOURS / FINAL_SAMPLING_HOURS))
final_pairs = select_pairs(final_sampled_train, FINAL_SAMPLING_HOURS, FINAL_WINDOW_BARS, FINAL_STRATEGY, FINAL_PARAMS)
print(f"Final train-only configuration: {FINAL_SAMPLING_HOURS}h, window={FINAL_WINDOW_HOURS:.0f}h ({FINAL_WINDOW_BARS} bars), {FINAL_STRATEGY}, weights={FINAL_WEIGHT_MODE}")
print(f"Mean train half-life: {FINAL_MEAN_HALF_LIFE_HOURS:.1f}h; selected pairs: {len(final_pairs)}")
display(final_pairs)

In [ ]:
# Untouched final test evaluation

# FINAL TEST: no pair re-selection, no parameter tuning, no weight re-estimation.
final_sampled_all = sample_market_data(hourly_prices, FINAL_SAMPLING_HOURS)
_, final_test = split_at_fixed_boundary(final_sampled_all)
final_rets = {}
final_train_rets = {}

# Warm up the test indicators with training observations, but only evaluate returns after TEST_START.
warmup = max(FINAL_WINDOW_BARS, VOLUME_WINDOW_HOURS // FINAL_SAMPLING_HOURS + 2)
final_input = pd.concat([final_sampled_train.tail(warmup), final_test], ignore_index=True).drop_duplicates("timestamp")
for _, row in final_pairs.iterrows():
    a, b = row["Asset_A"], row["Asset_B"]
    if FINAL_STRATEGY == "zscore":
        bt_train, _ = backtest_pair(final_sampled_train, a, b, row["Alpha"], row["Beta"], FINAL_STRATEGY, FINAL_WINDOW_BARS, FINAL_SAMPLING_HOURS, fee_bps=FEE_BPS, entry_z=FINAL_PARAMS["entry_z"], exit_z=FINAL_PARAMS["exit_z"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // FINAL_SAMPLING_HOURS))
        bt_all, _ = backtest_pair(final_input, a, b, row["Alpha"], row["Beta"], FINAL_STRATEGY, FINAL_WINDOW_BARS, FINAL_SAMPLING_HOURS, fee_bps=FEE_BPS, entry_z=FINAL_PARAMS["entry_z"], exit_z=FINAL_PARAMS["exit_z"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // FINAL_SAMPLING_HOURS))
    else:
        bt_train, _ = backtest_pair(final_sampled_train, a, b, row["Alpha"], row["Beta"], FINAL_STRATEGY, FINAL_WINDOW_BARS, FINAL_SAMPLING_HOURS, fee_bps=FEE_BPS, band_k=FINAL_PARAMS["band_k"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // FINAL_SAMPLING_HOURS))
        bt_all, _ = backtest_pair(final_input, a, b, row["Alpha"], row["Beta"], FINAL_STRATEGY, FINAL_WINDOW_BARS, FINAL_SAMPLING_HOURS, fee_bps=FEE_BPS, band_k=FINAL_PARAMS["band_k"], volume_window_bars=max(2, VOLUME_WINDOW_HOURS // FINAL_SAMPLING_HOURS))
    name = row["Pair"]
    final_train_rets[name] = bt_train["strategy_ret"].rename(name)
    final_rets[name] = bt_all.loc[bt_all.index >= TEST_START, "strategy_ret"].rename(name)

# Weights are estimated ON TRAIN ONLY.
weights = inverse_vol_weights(final_train_rets) if FINAL_WEIGHT_MODE == "equal_vol" else enhanced_weights(final_train_rets, final_pairs.set_index("Pair"))

# One frozen-weight portfolio over the untouched test period.
test_mat = pd.concat(final_rets, axis=1).dropna(how="all")
w = weights.reindex(test_mat.columns).fillna(0)
if w.sum() > 0: w = w / w.sum()
test_port = test_mat.mul(w, axis=1).sum(axis=1).dropna()
test_summary = portfolio_metrics(test_port, FINAL_SAMPLING_HOURS)
print("FINAL OUT-OF-SAMPLE RESULT (run once):")
print(test_summary)
print("\nFrozen test weights:")
print(w.sort_values(ascending=False))

In [ ]:
# Final descriptive diagnostics

# Optional diagnostics that use TEST returns only AFTER the strategy is frozen.
# These are descriptive; they must not feed back into configuration decisions.
print(f"Test bars: {len(test_port)}")
print(f"Test start: {test_port.index.min()}")
print(f"Test end:   {test_port.index.max()}")
print(f"Test Sharpe: {test_summary['Sharpe']:.3f}")
print(f"Test net return: {test_summary['NetReturn']:.3%}")
print(f"Test max drawdown: {test_summary['MaxDD']:.3%}")

plt.figure(figsize=(10, 5))
plt.plot((1 + test_port.fillna(0)).cumprod(), label=FINAL_STRATEGY)
plt.title(f"Frozen Out-of-Sample Equity Curve ({FINAL_SAMPLING_HOURS}h)")
plt.legend()
plt.tight_layout()
plt.show()